In [1]:
import os
import yaml
import multiprocessing as mp
import time
import tools21cm as t2c
from tqdm import tqdm
import astropy.units as u
import astropy.constants as cst
import cupy as cp
import numpy as np
import itertools


def telescope_beam_pattern(x, y, D, z):
    """Returns a 2D Gaussian array representing the telescope beam pattern.

    Parameters
    ----------
    x : ndarray
        Meshgrid x-coordinates.
    y : ndarray
        Meshgrid y-coordinates.
    D : float, optional
        Telescope dish diameter, in meters (default is 45).
    z : float, optional
        Redshift, which is converted to observation frequency (default is 7).

    Returns
    -------
    ndarray
        A 2D Gaussian array representing the telescope beam pattern.

    Notes
    -----
    The Gaussian array is calculated based on the provided dish diameter and redshift.
    """
    freq = t2c.cosmo.z_to_nu(z) * 10**6
    lam = cst.c.value / freq
    fwhm = 1.03 * lam / D
    theta_0 = 0.6 * fwhm
    return np.exp(-((x**2 + y**2) / theta_0**2))


def synchrotron_image(z, ncells, boxsize, seed=False):
    """Generate a synchrotron image.

    Parameters:
    - z (float): Redshift value.
    - ncells (int): Number of cells in each dimension.
    - boxsize (float): Size of the box in Mpc.
    - seed (bool, optional): Whether to use a fixed seed for random number generation. Default is False.

    Returns:
    - T_real (ndarray): The real part of the synchrotron image.
    - l_cb (ndarray): The angular scale of the synchrotron image.
    """
    if seed:
        np.random.seed(seed)

    X = np.random.normal(size=(ncells, ncells))
    Y = np.random.normal(size=(ncells, ncells))

    A150, beta = 5.13 * 1e-4, 2.34

    U_cb = (
        (np.mgrid[-ncells / 2 : ncells / 2, -ncells / 2 : ncells / 2] + 0.5)
        * t2c.cosmo.z_to_cdist(z)
        / boxsize
    )
    l_cb = 2 * np.pi * np.sqrt(U_cb[0, :, :] ** 2 + U_cb[1, :, :] ** 2)
    C_syn = A150 * (1000 / l_cb) ** beta
    solid_angle = boxsize**2 / t2c.cosmo.z_to_cdist(z) ** 2
    AA = np.sqrt(solid_angle * C_syn / 2)
    T_four = AA * (X + Y * 1j) * np.sqrt(2)
    T_real = np.abs(np.fft.ifft2(T_four))

    return T_real, l_cb


def visibility_matrix_func_new_method(
    xyz_coord, sky_coord, redshift, I_sky, beam_pattern
):
    """Calculates the visibility matrix, visibility list, and baseline
    coordinates for interferometry measurements.

    Parameters:
    - xyz_coord (numpy.ndarray): Array of shape (N_ant, 3) containing the x, y, and z coordinates of the antennas.
    - sky_coord (numpy.ndarray): Array of shape (N_sky,) containing the sky coordinates.
    - redshift (float): The redshift value.
    - I_sky (numpy.ndarray): Array of shape (N_sky,) containing the sky intensity values.
    - beam_pattern (numpy.ndarray): Array of shape (N_sky,) containing the beam pattern values.

    Returns:
    - visibility_list (numpy.ndarray): Array of shape (N_B,) containing the visibility values for each baseline.
    - visibility_matrix (numpy.ndarray): Array of shape (N_ant, N_ant) containing the visibility values for each antenna pair.
    - baseline_coord (numpy.ndarray): Array of shape (N_B, 3) containing the baseline coordinates.
    """
    xyz_coord = np.asarray(xyz_coord, dtype=np.float32)
    sky_coord = np.asarray(sky_coord, dtype=np.float32)
    I_sky = np.asarray(I_sky, dtype=np.float32)
    beam_pattern = np.asarray(beam_pattern, dtype=np.float32)
    
    l_coord, m_coord = np.meshgrid(sky_coord, sky_coord)
    N_ant = len(xyz_coord)
    N_B = int(N_ant * (N_ant - 1) / 2)
    freq = t2c.cosmo.z_to_nu(redshift) * 10**6
    lam = cst.c.value / freq
    pair_comb = list(itertools.combinations(range(N_ant), 2))
    baseline_coord = np.empty((N_B, xyz_coord.shape[1]), dtype= np.float32)
    visibility_matrix = np.zeros((N_ant, N_ant), dtype=np.complex64)
    visibility_list = np.empty(N_B, dtype=np.complex64)

    for i in tqdm(range(N_B)):
        aa, bb = pair_comb[i]
        uv = (xyz_coord[bb] - xyz_coord[aa]) / lam
        baseline_coord[i] = uv
        fringe = np.exp(-2j * np.pi * (uv[0] * l_coord + uv[1] * m_coord))
        vis = np.sum(fringe * I_sky * beam_pattern)
        visibility_list[i] = vis
        visibility_matrix[aa, bb] = vis
        visibility_matrix[bb, aa] = np.conj(vis)

    return visibility_list, visibility_matrix, baseline_coord


def visibility_matrix_func_cupy(xyz_coord, sky_coord, redshift, I_sky, beam_pattern):
    # Ensure all inputs are CuPy arrays with reduced precision
    xyz_coord = cp.asarray(xyz_coord, dtype=cp.float32)
    sky_coord = cp.asarray(sky_coord, dtype=cp.float32)
    I_sky = cp.asarray(I_sky, dtype=cp.float32)
    beam_pattern = cp.asarray(beam_pattern, dtype=cp.float32)

    l_coord, m_coord = cp.meshgrid(sky_coord, sky_coord)
    N_ant = xyz_coord.shape[0]
    N_B = N_ant * (N_ant - 1) // 2

    # Compute the frequency and wavelength
    freq = t2c.cosmo.z_to_nu(redshift) * 10**6
    lam = cst.c.value / freq

    # Calculate all pair combinations
    pairs = cp.array(list(itertools.combinations(range(N_ant), 2)), dtype=cp.int32)
    aa = pairs[:, 0]
    bb = pairs[:, 1]

    # Calculate baseline coordinates
    uv = (xyz_coord[bb] - xyz_coord[aa]) / lam
    baseline_coord = uv.astype(cp.float32)

    # Calculate fringes
    fringe = cp.exp(-2j * cp.pi * (uv[:, 0][:, cp.newaxis, cp.newaxis] * l_coord + uv[:, 1][:, cp.newaxis, cp.newaxis] * m_coord)).astype(cp.complex64)

    # Calculate visibilities
    visibilities = cp.sum(fringe * I_sky * beam_pattern, axis=(1, 2)).astype(cp.complex64)

    # Initialize visibility matrix
    visibility_matrix = cp.zeros((N_ant, N_ant), dtype=cp.complex64)
    
    # Populate visibility matrix
    visibility_matrix[aa, bb] = visibilities
    visibility_matrix[bb, aa] = cp.conj(visibilities)
    
    cp.cuda.Stream.null.synchronize()
    
    visibilities = cp.asnumpy(visibilities)
    visibility_matrix = cp.asnumpy(visibility_matrix)
    baseline_coord = cp.asnumpy(baseline_coord)

    return visibilities, visibility_matrix, baseline_coord


def filter_baselines(baselines, visibilities, u_freq, v_freq):
    """Filters baselines and corresponding visibilities based on the range of
    frequencies. This is so that the baselines outside the frequency range of
    the fft image are removed.

    Parameters:
    baselines (list): List of baseline coordinates.
    visibilities (list): List of corresponding visibilities.
    u_freq (list): List of u frequencies.
    v_freq (list): List of v frequencies.

    Returns:
    filtered_baselines (ndarray): Numpy array of filtered baseline coordinates.
    filtered_visibilities (ndarray): Numpy array of filtered visibilities.
    """
    filtered_baselines = []
    filtered_visibilities = []

    for baseline, visibility in zip(baselines, visibilities):
        x, y = baseline
        # Check if baseline coordinates are within the range of frequencies
        if (np.min(u_freq) <= x <= np.max(u_freq)) and (
            np.min(v_freq) <= y <= np.max(v_freq)
        ):
            filtered_baselines.append(baseline)
            filtered_visibilities.append(visibility)
    return np.array(filtered_baselines), np.array(filtered_visibilities)


def filter_baselines_cupy(baselines, visibilities, u_freq, v_freq):
    """Filters baselines and corresponding visibilities based on the range of
    frequencies. This is so that the baselines outside the frequency range of
    the fft image are removed.

    Parameters:
    baselines (ndarray): Numpy array of baseline coordinates.
    visibilities (ndarray): Numpy array of corresponding visibilities.
    u_freq (ndarray): Numpy array of u frequencies.
    v_freq (ndarray): Numpy array of v frequencies.

    Returns:
    filtered_baselines (ndarray): CuPy array of filtered baseline coordinates.
    filtered_visibilities (ndarray): CuPy array of filtered visibilities.
    """
    # Convert inputs to CuPy arrays
    baselines_cp = cp.array(baselines)
    visibilities_cp = cp.array(visibilities)
    u_freq_cp = cp.array(u_freq)
    v_freq_cp = cp.array(v_freq)
    
    # Find the min and max frequencies
    u_min, u_max = cp.min(u_freq_cp), cp.max(u_freq_cp)
    v_min, v_max = cp.min(v_freq_cp), cp.max(v_freq_cp)
    
    # Filter baselines based on frequency range
    x = baselines_cp[:, 0]
    y = baselines_cp[:, 1]
    mask = (u_min <= x) & (x <= u_max) & (v_min <= y) & (y <= v_max)
    
    # Apply mask to baselines and visibilities
    filtered_baselines = baselines_cp[mask]
    filtered_visibilities = visibilities_cp[mask]
    
    return filtered_baselines, filtered_visibilities


def random_layout(N_ant, range, seed):
    """Generates a random layout of antennas within the specified range.

    Parameters
    ----------
    N_ant : int
        Number of antennas to generate.
    range : tuple
        Tuple specifying the range of coordinates in which antennas will be placed.
        It should be in the format (min_value, max_value).
    seed : int
        Seed value for random number generation.

    Returns
    -------
    ndarray
        An array containing the coordinates of the randomly placed antennas.
        Each row represents the (x, y) coordinates of an antenna.
    """

    np.random.seed(seed)
    X = np.random.uniform(range[0], range[1], N_ant)
    Y = np.random.uniform(range[0], range[1], N_ant)

    return np.dstack((X, Y)).reshape(-1, 2)


def _splice_values_equal_bins_cupy(blcoord_sorted, blmag_sorted, vis_sorted, num_bins):
    """
    The provided arrays should be cupy arrays.
    """
    # Step 1: Calculate the number of elements per bin
    num_elements = len(blmag_sorted)
    elements_per_bin = num_elements // num_bins
    extra_elements = num_elements % num_bins

    # Step 2: Create an array of bin sizes
    bin_sizes = cp.full(num_bins, elements_per_bin)
    bin_sizes[:extra_elements] += 1

    # Step 3: Calculate the cumulative sum of bin sizes
    cumulative_sizes = cp.cumsum(bin_sizes)
    cumulative_sizes_np = cp.asnumpy(cumulative_sizes)  # Convert to NumPy array

    # Step 4: Initialize the bins
    blcoord_bins = cp.split(blcoord_sorted, cumulative_sizes_np[:-1])
    blmag_bins = cp.split(blmag_sorted, cumulative_sizes_np[:-1])
    vis_bins = cp.split(vis_sorted, cumulative_sizes_np[:-1])

    return blcoord_bins, blmag_bins, vis_bins

def _splice_values_function_fit_cupy(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins):
    bins = cp.linspace(np.min(blmag_sorted_transformed), np.max(blmag_sorted_transformed), num_bins)
    bin_indices = np.digitize(blmag_sorted_transformed, bins) - 1
    
    # Use histogram to count the occurrences of each bin
    hist, _ = cp.histogram(bin_indices, bins=cp.arange(0, num_bins + 1))

    # Calculate the cumulative counts to determine the indices where each bin's values start and end
    cum_counts = cp.cumsum(hist)
    cumulative_sizes_np = cp.asnumpy(cum_counts)  # Convert to NumPy array

    bin_blcoords = cp.split(blcoord_sorted, cumulative_sizes_np[:-1])
    bin_vis = cp.split(vis_sorted, cumulative_sizes_np[:-1])
    bin_blmags = cp.split(blmag_sorted, cumulative_sizes_np[:-1])

    return bin_blcoords, bin_vis, bin_blmags


def compute_vals_cupy(blcoord, vis, sigma_0, V_0):

    bin_len = len(blcoord)
    if bin_len == 0 or bin_len == 1:
        return 0, 0, 0

    # Create a meshgrid for vectorized pairwise operations
    idx_i, idx_j = np.triu_indices(bin_len, k=1)
    idx_i = cp.asarray(idx_i)
    idx_j = cp.asarray(idx_j)

    # Calculate pairwise distances
    delta= blcoord[idx_i] - blcoord[idx_j]
    distance_norm = cp.linalg.norm(delta, axis=1)
    
    combination_len_unfiltered= len(distance_norm)
    
    # Filter pairs where distance is less than or equal to sigma_0
    valid_pairs = distance_norm > sigma_0
    if not cp.any(valid_pairs):
        return 0, 0, 0

    distance_norm = distance_norm[valid_pairs]
    idx_i = idx_i[valid_pairs]
    idx_j = idx_j[valid_pairs]
    
    combination_len_filtered= len(distance_norm)
    print("Theoretical combinations: ", combination_len_unfiltered,
         " - When filtered: ", combination_len_filtered)
    
    # Calculate weights
    weight = cp.exp(-(distance_norm**2) / sigma_0**2)

    # Calculate the visibility product
    visibility_product = vis[idx_i] * cp.conj(vis[idx_j])

    #Calclating the bare estimator
    be_numerator = cp.sum(weight * visibility_product)
    be_denominator = cp.sum(weight * V_0 * cp.exp(-(distance_norm**2) / (sigma_0**2)))

    #Calculating the mean of ell
    l_i = 2 * cp.pi * cp.sqrt(blcoord[idx_i, 0]**2 + blcoord[idx_i, 1]**2)
    exp_weight_distance = weight * cp.exp(-(distance_norm**2) / (sigma_0**2))

    ell_numerator = cp.sum(exp_weight_distance * l_i)
    ell_denominator = cp.sum(exp_weight_distance)

    #Computing arrays needed for the error (variance)
    Eb_square_mean_numerator = cp.sum((exp_weight_distance * 5.13 * 1e-4 * (1000 / l_i) ** 2.34) ** 2)
    Eb_mean_square_numerator = cp.sum(exp_weight_distance * 5.13 * 1e-4 * (1000 / l_i) ** 2.34)
    Eb_denominator = cp.sum(exp_weight_distance)

    bare_estimator = cp.abs(be_numerator) / be_denominator
    ell_mean = ell_numerator / ell_denominator
    var_squared = (Eb_square_mean_numerator / Eb_denominator - (Eb_mean_square_numerator / Eb_denominator) ** 2)

    # Convert results back to numpy arrays
    bare_estimator = cp.asnumpy(bare_estimator)
    ell_mean = cp.asnumpy(ell_mean)
    var_squared = cp.asnumpy(var_squared)

    return bare_estimator, ell_mean, np.abs(var_squared)


def bare_estimator_func_v4(baselines, visibilities, rshift, D, num_bins, bin_type="log"):
    freq = t2c.cosmo.z_to_nu(rshift) * 10**6
    lam = cst.c.value / freq
    fwhm = 1.03 * lam / D
    sigma_0 = 0.76 / fwhm
    theta_0 = 0.6 * fwhm
    V_0 = np.pi * theta_0**2 / 2

    bl_mag = np.linalg.norm(baselines, axis=1)
    blcoord_blmag_vis = zip(baselines, bl_mag, visibilities)
    sorted_blcoord_blmag_vis = sorted(blcoord_blmag_vis, key=lambda x: x[1])
    blcoord_sorted, blmag_sorted, vis_sorted = zip(*sorted_blcoord_blmag_vis)
    print("Length of input arrays: ", len(blmag_sorted))

    if bin_type == "log":
        blmag_sorted_transformed = np.log10(blmag_sorted)
        bin_values = _splice_values_function_fit(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)
        
    elif bin_type == "linear":
        blmag_sorted_transformed = blmag_sorted
        bin_values = _splice_values_function_fit(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

    elif bin_type == "equal_length":
        bin_values = _splice_values_equal_bins(blcoord_sorted, blmag_sorted, vis_sorted, num_bins)
    else:
        raise ValueError("Unsupported bin_type")

    for key in bin_values.keys():
        bin_len = len(bin_values[key])
        print("Key: ", key, "; Key legnth: ", bin_len, "; Combinations: ", bin_len * (bin_len - 1) / 2)

    # create a pool of processes
    pool = mp.Pool(processes=num_bins)
    # map the function to the input data and calculate the results
    results = pool.starmap(
        compute_vals, [(bin_values[key], sigma_0, V_0) for key in bin_values.keys()]
    )
    pool.close()
    pool.join()

    # Combine the results from the processes
    bare_estimator = []
    ell_mean = []
    var_squared = []
    for result in results:
        bare_estimator.append(result[0])
        ell_mean.append(result[1])
        var_squared.append(result[2])

    np_bare_estimator = np.array(bare_estimator)
    np_ell_mean = np.array(ell_mean)
    np_var_squared = np.array(var_squared)

    # remove zero values from the arrays
    np_bare_estimator = np_bare_estimator[np_bare_estimator != 0]
    np_ell_mean = np_ell_mean[np_ell_mean != 0]
    np_var_squared = np_var_squared[np_var_squared != 0]

    return np_bare_estimator, np_ell_mean, np.abs(np_var_squared)


def bare_estimator_func_cupy(baselines, visibilities, rshift, D, num_bins, bin_type="log"):
    freq = t2c.cosmo.z_to_nu(rshift) * 10**6
    lam = cst.c.value / freq
    fwhm = 1.03 * lam / D
    sigma_0 = 0.76 / fwhm
    theta_0 = 0.6 * fwhm
    V_0 = np.pi * theta_0**2 / 2

    start_time = time.time()
    # Step 0: turn arrays into cupy arrays
    baselines= cp.asarray(baselines, dtype= cp.float32)
    visibilities= cp.asarray(visibilities, dtype= cp.complex64)

    # Step 1: Calculate the baseline magnitudes
    bl_mag = cp.linalg.norm(baselines, axis=1)

    # Stack baselines, magnitudes, and visibilities into a single array for sorting
    #combined_array = cp.hstack((baselines[:,0, None], baselines[:,1, None], bl_mag[:, None], visibilities[:, None]))
    combined_array = cp.hstack((baselines[:], bl_mag[:, None], visibilities[:, None]))

    # Sort the combined array based on the magnitudes
    sorted_combined_array = combined_array[cp.argsort(combined_array[:,2])]

    # Split the sorted array back into individual components
    blcoord_sorted = sorted_combined_array[:, 0:2]
    blcoord_sorted = blcoord_sorted.astype(cp.float32)  # Change to desired dtype
    blmag_sorted = sorted_combined_array[:, 2]
    blmag_sorted = blmag_sorted.astype(cp.float32)      # Change to desired dtype
    vis_sorted = sorted_combined_array[:, -1]
    vis_sorted = vis_sorted.astype(cp.complex64)          # Change to desired dtype

    print("Length of input arrays: ", len(blmag_sorted))

    if bin_type == "log":
        blmag_sorted_transformed = cp.log10(blmag_sorted)
        blcoord_bins, blmag_bins, vis_bins = _splice_values_function_fit_cupy(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

    elif bin_type == "linear":
        blmag_sorted_transformed = blmag_sorted
        blcoord_bins, blmag_bins, vis_bins = _splice_values_function_fit_cupy(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

    elif bin_type == "equal_length":
        blcoord_bins, blmag_bins, vis_bins = _splice_values_equal_bins_cupy(blcoord_sorted, blmag_sorted, vis_sorted, num_bins)
    else:
        raise ValueError("Unsupported bin_type")

    end_time = time.time()
    print('Time to sort and bin values: ', end_time-start_time, 's')

    # Verify the bins
    for i, bin in enumerate(blcoord_bins):
        print(f"Bin {i+1}: {len(bin)} elements - Combinations: {int(len(bin)*(len(bin)-1) / 2)}")
    # Print to verify the results
    print("Total elements after binning: ", cp.sum(cp.array([len(bin) for bin in blmag_bins])))

    bare_estimators = np.empty(num_bins, dtype=np.float32)
    average_ells = np.empty(num_bins, dtype=np.float32)
    square_variances = np.empty(num_bins, dtype=np.float32)

    start_time = time.time()
    for i in tqdm(range(num_bins)):
        BE, ell_avg, var_squared= compute_vals_cupy(blcoord_bins[i], vis_bins[i], sigma_0, V_0) 

        bare_estimators[i] = BE
        average_ells[i] = ell_avg
        square_variances[i] = var_squared
    end_time = time.time()
    print('Time to run the bare estimator: ', end_time-start_time, 's')
    
    return bare_estimators, average_ells, square_variances


def read_config(input_path):
    with open(input_path) as stream:
        try:
            config = yaml.safe_load(stream)
        except yaml.YAMLError as exc:
            print(exc)
    return config

In [2]:
def setup(config):
    
    # Determine which function to use based on the configuration
    estimator_func_choice = config.get('computation_type')
    if estimator_func_choice == 'cupy':
        bare_estimator_func = bare_estimator_func_cupy
        visibility_matrix_func = visibility_matrix_func_cupy
    elif estimator_func_choice == 'noMP':
        bare_estimator_func = bare_estimator_func_v4_noMP
        visibility_matrix_func = visibility_matrix_func_new_method
    else:
        bare_estimator_func = bare_estimator_func_v4
        visibility_matrix_func = visibility_matrix_func_new_method
    
    FoV = config["FoV"] * u.deg
    # define a 1D sky and get RA coordinate
    num_pix = config["num_pix"]
    thet = np.linspace(-FoV / 2, FoV / 2, num_pix).to("rad").value

    # Create a grid of points (we ignore third dimension, i.e. n-axis)
    range_meters = float(config["range_meters"])
    num_antennas = config["base_num_antennas"] ** config["exponent_num_antennas"]
    seed = config["seed"]
    l_coord, m_coord = np.meshgrid(thet, thet)
    #_ = np.dstack((l_coord, m_coord)).reshape(-1, 2)
    layout = random_layout(num_antennas, [-range_meters, range_meters], seed)

    # Angular diameter distance (in Mpc) for a specified redshift and FoV in deg
    rshift = config["redshift"]
    com_dist = FoV.to("rad").value * t2c.cosmo.z_to_cdist(rshift)

    # Synchrotron source, beam pattern
    antenna_diameter_meters = config["antenna_diameter_meters"]
    synchro_source, l_val = synchrotron_image(rshift, len(thet), com_dist, seed=seed)
    beam_pattern = telescope_beam_pattern(l_coord, m_coord, D=antenna_diameter_meters, z=rshift)

    # Compute visibilities, baselines and visibility matrix
    start_time = time.time()
    visls, vismx, bls = visibility_matrix_func(layout, thet, rshift, synchro_source, beam_pattern)
    end_time = time.time()
    # Calculate elapsed time
    execution_time = end_time - start_time
    print("Visibility func execution time:", execution_time, "seconds")

    
    # Computing the frequency range of the FFT image
    uu = np.fft.fftfreq(synchro_source.shape[0], np.diff(thet)[0])
    vv = np.fft.fftfreq(synchro_source.shape[1], np.diff(thet)[0])
    uu = np.fft.fftshift(uu)
    vv = np.fft.fftshift(vv)
    u_freq, v_freq = np.meshgrid(uu, vv)

    # Filter out baselines out of the frequency range of the image fft
    filter_bls, filter_vis = filter_baselines(bls, visls, u_freq, v_freq)

    # Compute the bare estimator
    start_time = time.time()
    binning_method = config["binning_method"]
    num_bins = config["num_bins"]
    bare_estimator, ell_mean, var_squared = bare_estimator_func(filter_bls, filter_vis, rshift, antenna_diameter_meters, num_bins, binning_method)
    end_time = time.time()
    # Calculate elapsed time
    execution_time = end_time - start_time
    print("Bare estimator func execution time:", execution_time, "seconds")
    return bare_estimator, ell_mean, var_squared

In [23]:
if __name__ == "__main__":
    config = read_config("input/config.yaml")
    bare_estimator, ell_mean, var_squared = setup(config)
    use_multiprocessing= config.get('use_multiprocessing', True)
    
    if use_multiprocessing:
        mp_suffix = "MP"
    else:
        mp_suffix = "NoMP"

    print("Bare estimator values:", bare_estimator)
    print("Ell mean values:", ell_mean)
    print("Variance squared values:", var_squared)
    print(len(bare_estimator), len(ell_mean), len(var_squared))

"""    filename = (
        f"BE_{config['FoV']}deg_Z{config['redshift']}_"
        f"Pix{config['num_pix']}_"
        f"Rng[m]{config['range_meters']}_"
        f"Ant{config['base_num_antennas']}"
        f"^{config['exponent_num_antennas']}_"
        f"D[m]{config['antenna_diameter_meters']}_"
        f"Bins{config['num_bins']}_"
        f"{config['binning_method']}_"
        f"{config['img_dpi']}dpi_"
        f"{mp_suffix}.png")
    full_file_path = os.path.join(os.path.dirname("output/"), filename)
    plot(bare_estimator, ell_mean, var_squared, file_path=full_file_path, img_dpi=config['img_dpi'])"""

Visibility func execution time: 0.08198761940002441 seconds
Length of input arrays:  4808
Time to sort and bin values:  0.0012459754943847656 s
Bin 1: 241 elements - Combinations: 28920
Bin 2: 241 elements - Combinations: 28920
Bin 3: 241 elements - Combinations: 28920
Bin 4: 241 elements - Combinations: 28920
Bin 5: 241 elements - Combinations: 28920
Bin 6: 241 elements - Combinations: 28920
Bin 7: 241 elements - Combinations: 28920
Bin 8: 241 elements - Combinations: 28920
Bin 9: 240 elements - Combinations: 28680
Bin 10: 240 elements - Combinations: 28680
Bin 11: 240 elements - Combinations: 28680
Bin 12: 240 elements - Combinations: 28680
Bin 13: 240 elements - Combinations: 28680
Bin 14: 240 elements - Combinations: 28680
Bin 15: 240 elements - Combinations: 28680
Bin 16: 240 elements - Combinations: 28680
Bin 17: 240 elements - Combinations: 28680
Bin 18: 240 elements - Combinations: 28680
Bin 19: 240 elements - Combinations: 28680
Bin 20: 240 elements - Combinations: 28680
Total

100%|██████████| 20/20 [00:00<00:00, 356.24it/s]

Theoretical combinations:  28920  - When filtered:  28710
Theoretical combinations:  28920  - When filtered:  28734
Theoretical combinations:  28920  - When filtered:  28765
Theoretical combinations:  28920  - When filtered:  28782
Theoretical combinations:  28920  - When filtered:  28805
Theoretical combinations:  28920  - When filtered:  28793
Theoretical combinations:  28920  - When filtered:  28797
Theoretical combinations:  28920  - When filtered:  28816
Theoretical combinations:  28680  - When filtered:  28576
Theoretical combinations:  28680  - When filtered:  28582
Theoretical combinations:  28680  - When filtered:  28585
Theoretical combinations:  28680  - When filtered:  28614
Theoretical combinations:  28680  - When filtered:  28597
Theoretical combinations:  28680  - When filtered:  28603
Theoretical combinations:  28680  - When filtered:  28631
Theoretical combinations:  28680  - When filtered:  28635
Theoretical combinations:  28680  - When filtered:  28639
Theoretical co

'    filename = (\n        f"BE_{config[\'FoV\']}deg_Z{config[\'redshift\']}_"\n        f"Pix{config[\'num_pix\']}_"\n        f"Rng[m]{config[\'range_meters\']}_"\n        f"Ant{config[\'base_num_antennas\']}"\n        f"^{config[\'exponent_num_antennas\']}_"\n        f"D[m]{config[\'antenna_diameter_meters\']}_"\n        f"Bins{config[\'num_bins\']}_"\n        f"{config[\'binning_method\']}_"\n        f"{config[\'img_dpi\']}dpi_"\n        f"{mp_suffix}.png")\n    full_file_path = os.path.join(os.path.dirname("output/"), filename)\n    plot(bare_estimator, ell_mean, var_squared, file_path=full_file_path, img_dpi=config[\'img_dpi\'])'

In [3]:
# Determine which function to use based on the configuration
config = read_config("input/config.yaml")
estimator_func_choice = config.get('computation_type')
if estimator_func_choice == 'cupy':
    bare_estimator_func = bare_estimator_func_cupy
    visibility_matrix_func = visibility_matrix_func_cupy
elif estimator_func_choice == 'noMP':
    bare_estimator_func = bare_estimator_func_v4_noMP
    visibility_matrix_func = visibility_matrix_func_new_method
else:
    bare_estimator_func = bare_estimator_func_v4
    visibility_matrix_func = visibility_matrix_func_new_method

FoV = config["FoV"] * u.deg
# define a 1D sky and get RA coordinate
num_pix = config["num_pix"]
thet = np.linspace(-FoV / 2, FoV / 2, num_pix).to("rad").value

# Create a grid of points (we ignore third dimension, i.e. n-axis)
range_meters = float(config["range_meters"])
num_antennas = config["base_num_antennas"] ** config["exponent_num_antennas"]
seed = config["seed"]
l_coord, m_coord = np.meshgrid(thet, thet)
layout = random_layout(num_antennas, [-range_meters, range_meters], seed)

# Angular diameter distance (in Mpc) for a specified redshift and FoV in deg
rshift = config["redshift"]
com_dist = FoV.to("rad").value * t2c.cosmo.z_to_cdist(rshift)

# Synchrotron source, beam pattern
antenna_diameter_meters = config["antenna_diameter_meters"]
synchro_source, l_val = synchrotron_image(rshift, len(thet), com_dist, seed=seed)
beam_pattern = telescope_beam_pattern(l_coord, m_coord, D=antenna_diameter_meters, z=rshift)

# Compute visibilities, baselines and visibility matrix
start_time = time.time()
visls, vismx, bls = visibility_matrix_func(layout, thet, rshift, synchro_source, beam_pattern)
end_time = time.time()
# Calculate elapsed time
execution_time = end_time - start_time
print("Visibility func execution time:", execution_time, "seconds")


# Computing the frequency range of the FFT image
uu = np.fft.fftfreq(synchro_source.shape[0], np.diff(thet)[0])
vv = np.fft.fftfreq(synchro_source.shape[1], np.diff(thet)[0])
uu = np.fft.fftshift(uu)
vv = np.fft.fftshift(vv)
u_freq, v_freq = np.meshgrid(uu, vv)

# Filter out baselines out of the frequency range of the image fft
filter_bls, filter_vis = filter_baselines(bls, visls, u_freq, v_freq)

"""
# Compute the bare estimator
start_time = time.time()
binning_method = config["binning_method"]
num_bins = config["num_bins"]
bare_estimator, ell_mean, var_squared = bare_estimator_func(filter_bls, filter_vis, rshift, antenna_diameter_meters, num_bins, binning_method)
end_time = time.time()
# Calculate elapsed time
execution_time = end_time - start_time
print("Bare estimator func execution time:", execution_time, "seconds")
return bare_estimator, ell_mean, var_squared"""

Visibility func execution time: 1.0920724868774414 seconds


'\n# Compute the bare estimator\nstart_time = time.time()\nbinning_method = config["binning_method"]\nnum_bins = config["num_bins"]\nbare_estimator, ell_mean, var_squared = bare_estimator_func(filter_bls, filter_vis, rshift, antenna_diameter_meters, num_bins, binning_method)\nend_time = time.time()\n# Calculate elapsed time\nexecution_time = end_time - start_time\nprint("Bare estimator func execution time:", execution_time, "seconds")\nreturn bare_estimator, ell_mean, var_squared'

In [5]:
print(filter_bls.shape)
print(filter_vis.shape)

(4808, 2)
(4808,)


In [20]:
baselines= filter_bls
visibilities= filter_vis
D = config["antenna_diameter_meters"]
bin_type= 'equal_length'
num_bins=20

freq = t2c.cosmo.z_to_nu(rshift) * 10**6
lam = cst.c.value / freq
fwhm = 1.03 * lam / D
sigma_0 = 0.76 / fwhm
theta_0 = 0.6 * fwhm
V_0 = np.pi * theta_0**2 / 2

start_time = time.time()
# Step 0: turn arrays into cupy arrays
baselines= cp.asarray(baselines, dtype= cp.float32)
visibilities= cp.asarray(visibilities, dtype= cp.complex64)

# Step 1: Calculate the baseline magnitudes
bl_mag = cp.linalg.norm(baselines, axis=1)

# Stack baselines, magnitudes, and visibilities into a single array for sorting
#combined_array = cp.hstack((baselines[:,0, None], baselines[:,1, None], bl_mag[:, None], visibilities[:, None]))
combined_array = cp.hstack((baselines[:], bl_mag[:, None], visibilities[:, None]))

# Sort the combined array based on the magnitudes
sorted_combined_array = combined_array[cp.argsort(combined_array[:,2])]

# Split the sorted array back into individual components
blcoord_sorted = sorted_combined_array[:, 0:2]
blcoord_sorted = blcoord_sorted.astype(cp.float32)  # Change to desired dtype
blmag_sorted = sorted_combined_array[:, 2]
blmag_sorted = blmag_sorted.astype(cp.float32)      # Change to desired dtype
vis_sorted = sorted_combined_array[:, -1]
vis_sorted = vis_sorted.astype(cp.complex64)          # Change to desired dtype

print("Length of input arrays: ", len(blmag_sorted))

if bin_type == "log":
    blmag_sorted_transformed = cp.log10(blmag_sorted)
    blcoord_bins, blmag_bins, vis_bins = _splice_values_function_fit_cupy(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

elif bin_type == "linear":
    blmag_sorted_transformed = blmag_sorted
    blcoord_bins, blmag_bins, vis_bins = _splice_values_function_fit_cupy(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

elif bin_type == "equal_length":
    blcoord_bins, blmag_bins, vis_bins = _splice_values_equal_bins_cupy(blcoord_sorted, blmag_sorted, vis_sorted, num_bins)
else:
    raise ValueError("Unsupported bin_type")

end_time = time.time()
print('Time to sort and bin values: ', end_time-start_time, 's')

# Verify the bins
for i, bin in enumerate(blcoord_bins):
    print(f"Bin {i+1}: {len(bin)} elements - Combinations: {int(len(bin)*(len(bin)-1) / 2)}")
# Print to verify the results
print("Total elements after binning: ", cp.sum(cp.array([len(bin) for bin in blmag_bins])))

bare_estimators = np.empty(num_bins, dtype=np.float32)
average_ells = np.empty(num_bins, dtype=np.float32)
square_variances = np.empty(num_bins, dtype=np.float32)

start_time = time.time()
for i in tqdm(range(num_bins)):
    BE, ell_avg, var_squared= compute_vals_cupy(blcoord_bins[i], vis_bins[i], sigma_0, V_0) 
    
    bare_estimators[i] = BE
    average_ells[i] = ell_avg
    square_variances[i] = var_squared
end_time = time.time()
print('Time to run the bare estimator: ', end_time-start_time, 's')
"""
np_bare_estimator = np.array(bare_estimator)
np_ell_mean = np.array(ell_mean)
np_var_squared = np.array(var_squared)

# remove zero values from the arrays
#np_bare_estimator = np_bare_estimator[np_bare_estimator != 0]
#np_ell_mean = np_ell_mean[np_ell_mean != 0]
#np_var_squared = np_var_squared[np_var_squared != 0]

return cp.asnumpy(np_bare_estimator), cp.asnumpy(np_ell_mean), cp.abs(cp.asnumpy(np_var_squared))"""

Length of input arrays:  4808
Time to sort and bin values:  0.0029401779174804688 s
Bin 1: 241 elements - Combinations: 28920
Bin 2: 241 elements - Combinations: 28920
Bin 3: 241 elements - Combinations: 28920
Bin 4: 241 elements - Combinations: 28920
Bin 5: 241 elements - Combinations: 28920
Bin 6: 241 elements - Combinations: 28920
Bin 7: 241 elements - Combinations: 28920
Bin 8: 241 elements - Combinations: 28920
Bin 9: 240 elements - Combinations: 28680
Bin 10: 240 elements - Combinations: 28680
Bin 11: 240 elements - Combinations: 28680
Bin 12: 240 elements - Combinations: 28680
Bin 13: 240 elements - Combinations: 28680
Bin 14: 240 elements - Combinations: 28680
Bin 15: 240 elements - Combinations: 28680
Bin 16: 240 elements - Combinations: 28680
Bin 17: 240 elements - Combinations: 28680
Bin 18: 240 elements - Combinations: 28680
Bin 19: 240 elements - Combinations: 28680
Bin 20: 240 elements - Combinations: 28680
Total elements after binning:  4808


100%|██████████| 20/20 [00:00<00:00, 321.35it/s]

Theoretical combinations:  28920  - When filtered:  28710
Theoretical combinations:  28920  - When filtered:  28734
Theoretical combinations:  28920  - When filtered:  28765
Theoretical combinations:  28920  - When filtered:  28782
Theoretical combinations:  28920  - When filtered:  28805
Theoretical combinations:  28920  - When filtered:  28793
Theoretical combinations:  28920  - When filtered:  28797
Theoretical combinations:  28920  - When filtered:  28816
Theoretical combinations:  28680  - When filtered:  28576
Theoretical combinations:  28680  - When filtered:  28582
Theoretical combinations:  28680  - When filtered:  28585
Theoretical combinations:  28680  - When filtered:  28614
Theoretical combinations:  28680  - When filtered:  28597
Theoretical combinations:  28680  - When filtered:  28603
Theoretical combinations:  28680  - When filtered:  28631
Theoretical combinations:  28680  - When filtered:  28635
Theoretical combinations:  28680  - When filtered:  28639
Theoretical co

'\nnp_bare_estimator = np.array(bare_estimator)\nnp_ell_mean = np.array(ell_mean)\nnp_var_squared = np.array(var_squared)\n\n# remove zero values from the arrays\n#np_bare_estimator = np_bare_estimator[np_bare_estimator != 0]\n#np_ell_mean = np_ell_mean[np_ell_mean != 0]\n#np_var_squared = np_var_squared[np_var_squared != 0]\n\nreturn cp.asnumpy(np_bare_estimator), cp.asnumpy(np_ell_mean), cp.abs(cp.asnumpy(np_var_squared))'

In [ ]:
def bare_estimator_func_cupy(baselines, visibilities, rshift, D, num_bins, bin_type="log"):
    freq = t2c.cosmo.z_to_nu(rshift) * 10**6
    lam = cst.c.value / freq
    fwhm = 1.03 * lam / D
    sigma_0 = 0.76 / fwhm
    theta_0 = 0.6 * fwhm
    V_0 = np.pi * theta_0**2 / 2

    start_time = time.time()
    # Step 0: turn arrays into cupy arrays
    baselines= cp.asarray(baselines, dtype= cp.float32)
    visibilities= cp.asarray(visibilities, dtype= cp.complex64)

    # Step 1: Calculate the baseline magnitudes
    bl_mag = cp.linalg.norm(baselines, axis=1)

    # Stack baselines, magnitudes, and visibilities into a single array for sorting
    #combined_array = cp.hstack((baselines[:,0, None], baselines[:,1, None], bl_mag[:, None], visibilities[:, None]))
    combined_array = cp.hstack((baselines[:], bl_mag[:, None], visibilities[:, None]))

    # Sort the combined array based on the magnitudes
    sorted_combined_array = combined_array[cp.argsort(combined_array[:,2])]

    # Split the sorted array back into individual components
    blcoord_sorted = sorted_combined_array[:, 0:2]
    blcoord_sorted = blcoord_sorted.astype(cp.float32)  # Change to desired dtype
    blmag_sorted = sorted_combined_array[:, 2]
    blmag_sorted = blmag_sorted.astype(cp.float32)      # Change to desired dtype
    vis_sorted = sorted_combined_array[:, -1]
    vis_sorted = vis_sorted.astype(cp.complex64)          # Change to desired dtype

    print("Length of input arrays: ", len(blmag_sorted))

    if bin_type == "log":
        blmag_sorted_transformed = cp.log10(blmag_sorted)
        blcoord_bins, blmag_bins, vis_bins = _splice_values_function_fit_cupy(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

    elif bin_type == "linear":
        blmag_sorted_transformed = blmag_sorted
        blcoord_bins, blmag_bins, vis_bins = _splice_values_function_fit_cupy(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

    elif bin_type == "equal_length":
        blcoord_bins, blmag_bins, vis_bins = _splice_values_equal_bins_cupy(blcoord_sorted, blmag_sorted, vis_sorted, num_bins)
    else:
        raise ValueError("Unsupported bin_type")

    end_time = time.time()
    print('Time to sort and bin values: ', end_time-start_time, 's')

    # Verify the bins
    for i, bin in enumerate(blcoord_bins):
        print(f"Bin {i+1}: {len(bin)} elements - Combinations: {int(len(bin)*(len(bin)-1) / 2)}")
    # Print to verify the results
    print("Total elements after binning: ", cp.sum(cp.array([len(bin) for bin in blmag_bins])))

    bare_estimators = np.empty(num_bins, dtype=np.float32)
    average_ells = np.empty(num_bins, dtype=np.float32)
    square_variances = np.empty(num_bins, dtype=np.float32)

    start_time = time.time()
    for i in tqdm(range(num_bins)):
        BE, ell_avg, var_squared= compute_vals_cupy(blcoord_bins[i], vis_bins[i], sigma_0, V_0) 

        bare_estimators[i] = BE
        average_ells[i] = ell_avg
        square_variances[i] = var_squared
    end_time = time.time()
    print('Time to run the bare estimator: ', end_time-start_time, 's')
    
    return bare_estimators, average_ells, square_variances